In [1]:
import os, cv2, math, time, json, random, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F, torchvision
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
cv2.setNumThreads(0)

ROOT    = '/kaggle/input/datasets/ashery/chexpert/'
PARQUET = '/kaggle/input/datasets/shauryaveernagar/cleanfolds/train_clean_folds.parquet'
LABELS  = ['Cardiomegaly', 'Pleural Effusion', 'Atelectasis']
SIZE, BATCH, VAL_FOLD = 320, 64, 0          # BATCH 64 = 32 per T4
MEAN, STD = 0.5063, 0.2895

print("GPUs:", torch.cuda.device_count(), "| CPUs:", os.cpu_count())

df = pd.read_parquet(PARQUET)
if 'img_path' not in df.columns:
    df['img_path'] = df['Path'].apply(
        lambda p: p if os.path.isabs(p)
        else os.path.join(ROOT, p.replace('CheXpert-v1.0-small/', '', 1)))
df = df[~df['Path'].str.contains(r'(^|/)\._', regex=True, na=False)]

rate = df.sample(2000, random_state=0).img_path.map(os.path.exists).mean()
print("exists rate (2k sample):", rate)
assert rate > 0.99, "path problem, check ROOT / prefix"

GPUs: 2 | CPUs: 4


/tmp/ipykernel_50/583292655.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df['Path'].str.contains(r'(^|/)\._', regex=True, na=False)]


exists rate (2k sample): 1.0


In [8]:
from concurrent.futures import ThreadPoolExecutor
import shutil

S = 256
CACHE_DIR = "/tmp/cxr_cache"                 # always exists and is writable
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE = f"{CACHE_DIR}/cxr{S}.npy"

df = df.reset_index(drop=True)
df["cidx"] = np.arange(len(df))

need_gb = len(df) * S * S / 1e9
print(f"cache needs {need_gb:.1f} GB | free in {CACHE_DIR}: {shutil.disk_usage(CACHE_DIR).free/1e9:.1f} GB")

def load(p):
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None: return np.zeros((S, S), np.uint8)
    return cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA)

def cache_ok():
    if not os.path.exists(CACHE): return False
    try: return np.load(CACHE, mmap_mode="r").shape == (len(df), S, S)
    except Exception: return False

if not cache_ok():
    cache = np.lib.format.open_memmap(CACHE, mode="w+", dtype=np.uint8, shape=(len(df), S, S))
    t = time.time()
    with ThreadPoolExecutor(8) as ex:
        for i, a in enumerate(ex.map(load, df.img_path.values)):
            cache[i] = a
            if i % 10000 == 0: print(i, f"{time.time()-t:.0f}s", flush=True)
    cache.flush(); del cache
print("cache ready", np.load(CACHE, mmap_mode="r").shape)

cache needs 12.5 GB | free in /tmp/cxr_cache: 1126.0 GB
0 4s
10000 133s
20000 270s
30000 415s
40000 556s
50000 698s
60000 838s
70000 976s
80000 1114s
90000 1249s
100000 1383s
110000 1520s
120000 1648s
130000 1777s
140000 1909s
150000 2044s
160000 2187s
170000 2341s
180000 2512s
190000 2689s
cache ready (191027, 256, 256)


In [12]:
SIZE = S

class CXRDataset(Dataset):
    def __init__(self, df, policy="ignore"):
        assert policy in {"ignore", "zeros", "ones"}
        self.cidx = df["cidx"].values
        raw = np.nan_to_num(df[LABELS].values.astype(np.float32), nan=0.0)
        unc = raw == -1
        if policy == "zeros": raw[unc] = 0.0
        elif policy == "ones": raw[unc] = 1.0
        self.mask   = (~unc if policy == "ignore" else np.ones_like(raw, bool)).astype(np.float32)
        self.labels = np.where(raw == -1, 0.0, raw).astype(np.float32)
        self._mm = None

    def __len__(self): return len(self.cidx)

    def __getitem__(self, i):
        if self._mm is None:
            self._mm = np.load(CACHE, mmap_mode="r")
        img = np.array(self._mm[self.cidx[i]])
        return (torch.from_numpy(img)[None],
                torch.from_numpy(self.labels[i]), torch.from_numpy(self.mask[i]))

def make_loaders(policy="ignore", val_fold=VAL_FOLD, num_workers=4, frac=1.0):
    tr_df = df[df.fold != val_fold]
    if frac < 1.0: tr_df = tr_df.sample(frac=frac, random_state=0)
    tr_df = tr_df.reset_index(drop=True)
    tr = CXRDataset(tr_df, policy)
    pos = (tr.labels * tr.mask).sum(0)
    neg = ((1 - tr.labels) * tr.mask).sum(0)
    pos_weight = torch.tensor(neg / np.maximum(pos, 1), dtype=torch.float32)
    dl = DataLoader(tr, BATCH, shuffle=True, drop_last=True, num_workers=num_workers,
                    pin_memory=True, persistent_workers=True, prefetch_factor=4)
    return dl, pos_weight

class MaskedBCE(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")
    def forward(self, logits, y, m):
        return (self.bce(logits, y) * m).sum() / m.sum().clamp(min=1)

In [13]:
dev = "cuda"

def gpu_prep(x, train):
    x = x.float().div_(255)                                   # (B,1,H,W)
    if train:
        B = x.size(0); r = lambda s: (torch.rand(B, device=x.device) * 2 - 1) * s
        ang, sc = r(math.radians(7)), 1 + r(0.07)
        cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
        tx, ty = r(0.08), r(0.08)
        theta = torch.stack([torch.stack([cos, -sin, tx], 1),
                             torch.stack([sin,  cos, ty], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False),
                          padding_mode="zeros", align_corners=False)
        b = (1 + r(0.15)).view(B, 1, 1, 1); c = (1 + r(0.15)).view(B, 1, 1, 1)
        mu = x.mean((1, 2, 3), keepdim=True)
        x = (((x - mu) * c + mu) * b).clamp_(0, 1)
    x = (x - MEAN) / STD
    return x.expand(-1, 3, -1, -1).contiguous(memory_format=torch.channels_last)

In [14]:
import gc
def log(*a): print(*a, flush=True)

class AmpWrap(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        with torch.autocast("cuda", dtype=torch.float16):
            return self.m(x)

def run(POLICY, SEED=42, EPOCHS=3, PATIENCE=2, MIN_DELTA=1e-3,
        LR_BACKBONE=1e-4, LR_HEAD=1e-3, WD=1e-2, CLIP=1.0, LOG_SECS=30, USE_DP=True, FRAC=1.0):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    OUT = f"/kaggle/working/densenet121_{POLICY}_fold{VAL_FOLD}_seed{SEED}"; os.makedirs(OUT, exist_ok=True)

    # ---- data ----
    tr_dl, pw = make_loaders(POLICY, frac=FRAC)
    va_df = df[df.fold == VAL_FOLD].reset_index(drop=True)
    va_ds = CXRDataset(va_df, "ignore")
    va_dl = DataLoader(va_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True,
                       persistent_workers=True, prefetch_factor=4)

    ds = tr_dl.dataset
    vpos, vneg = (va_ds.labels * va_ds.mask).sum(0), ((1 - va_ds.labels) * va_ds.mask).sum(0)
    log(f"\n===== policy={POLICY} | train {len(ds)} imgs, {len(tr_dl)} steps | val {len(va_ds)} imgs =====")
    log(f"train masked entries/label : {(1 - ds.mask).sum(0).astype(int)}")
    log(f"train pos rate (unmasked)  : {((ds.labels * ds.mask).sum(0) / ds.mask.sum(0)).round(3)}")
    log(f"pos_weight                 : {pw.numpy().round(2)}")
    log(f"val certain pos / neg      : {vpos.astype(int)} / {vneg.astype(int)}  (tiny pos count = noisy metric)")

    # ---- model ----
    base = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.DEFAULT)
    base.classifier = nn.Linear(base.classifier.in_features, len(LABELS))
    base = base.to(dev).to(memory_format=torch.channels_last)
    groups = [dict(params=[p], lr=LR_HEAD if n.startswith("classifier") else LR_BACKBONE,
                   weight_decay=0.0 if p.ndim == 1 else WD) for n, p in base.named_parameters()]
    opt    = torch.optim.AdamW(groups)
    model  = nn.DataParallel(AmpWrap(base)) if (USE_DP and torch.cuda.device_count() > 1) else AmpWrap(base)
    crit   = MaskedBCE(pw.to(dev))
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    json.dump(dict(policy=POLICY, seed=SEED, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD, wd=WD, epochs=EPOCHS,
                   patience=PATIENCE, size=SIZE, batch=BATCH, val_fold=VAL_FOLD, frac=FRAC,
                   use_dp=USE_DP, pos_weight=pw.tolist()),
              open(f"{OUT}/config.json", "w"), indent=2)

    @torch.no_grad()
    def evaluate():
        model.eval(); P, Y, M, L = [], [], [], 0.0
        for x, y, m in va_dl:
            x, y, m = gpu_prep(x.to(dev, non_blocking=True), False), y.to(dev), m.to(dev)
            lg = model(x).float()
            L += crit(lg, y, m).item() * len(x)
            P.append(torch.sigmoid(lg).cpu()); Y.append(y.cpu()); M.append(m.cpu())
        P, Y, M = (torch.cat(t).numpy() for t in (P, Y, M))
        au, ap = [], []
        for j in range(len(LABELS)):
            k = M[:, j] == 1
            if len(np.unique(Y[k, j])) < 2: au.append(np.nan); ap.append(np.nan); continue
            au.append(roc_auc_score(Y[k, j], P[k, j])); ap.append(average_precision_score(Y[k, j], P[k, j]))
        return L / len(va_ds), np.array(au), np.array(ap), (P, Y, M)

    def ckpt(ep, best):
        return {"model": base.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": ep, "best": best}

    # ---- train ----
    hist, best, best_ep, bad, T0 = [], -1.0, 0, 0, time.time()
    steps = len(tr_dl)
    for ep in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); tlog = time.time()
        e_loss = torch.zeros((), device=dev); e_n = 0
        i_loss = torch.zeros((), device=dev); i_n = 0
        i_pos  = torch.zeros(len(LABELS), device=dev); i_msk = torch.zeros(len(LABELS), device=dev)
        i_gmax = torch.zeros((), device=dev); i_bad = torch.zeros((), device=dev)

        for i, (x, y, m) in enumerate(tr_dl, 1):
            x = gpu_prep(x.to(dev, non_blocking=True), True)
            y, m = y.to(dev, non_blocking=True), m.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x).float(), y, m)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(base.parameters(), CLIP)
            scaler.step(opt); scaler.update()

            b = len(x); ld = loss.detach() * b
            e_loss += ld; e_n += b; i_loss += ld; i_n += b
            i_pos += (y * m).sum(0); i_msk += m.sum(0)
            fin = torch.isfinite(gn)                                   # no Python `if` -> no GPU sync
            i_gmax = torch.where(fin, torch.maximum(i_gmax, gn.detach()), i_gmax)
            i_bad += (~fin).float()

            if i == 1 or time.time() - tlog >= LOG_SECS:
                dt = time.time() - tlog; el = time.time() - t0
                log(f"  ep{ep} {i}/{steps} ({100*i/steps:.0f}%) "
                    f"loss(last {i_n} imgs) {i_loss.item()/i_n:.4f} | run {e_loss.item()/e_n:.4f} | "
                    f"{(i_n/dt if i > 1 else 0):.0f} img/s | lr_head {opt.param_groups[-1]['lr']:.2e} | "
                    f"gnorm max {i_gmax.item():.2f} (clip {CLIP}) | inf-steps {int(i_bad.item())} | "
                    f"amp scale {scaler.get_scale():.0f} | "
                    f"batch pos% {(100*i_pos/i_msk.clamp(min=1)).cpu().numpy().round(1)} "
                    f"| mem {torch.cuda.max_memory_allocated()/1e9:.1f}GB | "
                    f"el {el:.0f}s eta {el/i*(steps-i):.0f}s")
                tlog = time.time(); i_loss.zero_(); i_n = 0; i_pos.zero_(); i_msk.zero_(); i_gmax.zero_(); i_bad.zero_()
        sched.step()
        t_train = time.time() - t0; tr_loss = e_loss.item() / e_n

        t1 = time.time(); vl, au, ap, (P, Y, M) = evaluate(); t_eval = time.time() - t1
        row = dict(epoch=ep, train_loss=tr_loss, val_loss=vl, mean_auroc=np.nanmean(au),
                   mean_auprc=np.nanmean(ap), sec=time.time() - t0,
                   **{f"auroc_{l}": a for l, a in zip(LABELS, au)},
                   **{f"auprc_{l}": a for l, a in zip(LABELS, ap)})
        hist.append(row); pd.DataFrame(hist).to_csv(f"{OUT}/metrics.csv", index=False)

        improved = row["mean_auroc"] > best + MIN_DELTA
        log(f"ep{ep}/{EPOCHS} [{POLICY}] train {tr_loss:.4f} val {vl:.4f} (gap {vl-tr_loss:+.4f}) | "
            f"mean AUROC {row['mean_auroc']:.4f} AUPRC {row['mean_auprc']:.4f}")
        for l, a, p in zip(LABELS, au, ap):
            log(f"     {l:<17} AUROC {a:.3f}  AUPRC {p:.3f}")
        log(f"     time: train {t_train:.0f}s + eval {t_eval:.0f}s | total {(time.time()-T0)/60:.1f} min | "
            f"{'NEW BEST' if improved else f'no gain {bad+1}/{PATIENCE}'}")

        if improved:
            best, best_ep, bad = row["mean_auroc"], ep, 0
            torch.save(ckpt(ep, best), f"{OUT}/best.pt")
            cols = [c for c in ["Path", "img_path", "patient", "fold"] if c in va_df.columns]
            pred = va_df[cols].copy()
            for j, l in enumerate(LABELS):
                pred[f"p_{l}"], pred[f"y_{l}"], pred[f"m_{l}"] = P[:, j], Y[:, j], M[:, j]
            pred.to_parquet(f"{OUT}/val_preds.parquet", index=False)
        else:
            bad += 1
        torch.save(ckpt(ep, best), f"{OUT}/last.pt")
        if not improved and bad >= PATIENCE:
            log(f"early stop at ep{ep} (best ep{best_ep}, mean AUROC {best:.4f})"); break

    log(f"[{POLICY}] best mean AUROC {best:.4f} @ ep{best_ep}")
    out = pd.DataFrame(hist)
    del tr_dl, va_dl, model, base, opt, sched, scaler, crit
    gc.collect(); torch.cuda.empty_cache()
    return out

In [15]:
# 1) smoke test (~minutes): 1 epoch on 10% of train
_ = run("ignore", EPOCHS=1, FRAC=0.1)



===== policy=ignore | train 15173 imgs, 237 steps | val 39300 imgs =====
train masked entries/label : [ 531  778 2331]
train pos rate (unmasked)  : [0.131 0.429 0.187]
pos_weight                 : [6.63 1.33 4.35]
val certain pos / neg      : [ 4749 15920  6047] / [33128 21412 27286]  (tiny pos count = noisy metric)
  ep1 1/237 (0%) loss(last 64 imgs) 1.0742 | run 1.0742 | 0 img/s | lr_head 1.00e-03 | gnorm max 6.84 (clip 1.0) | inf-steps 0 | amp scale 65536 | batch pos% [10.  55.6 24.1] | mem 8.9GB | el 7s eta 1676s
  ep1 31/237 (13%) loss(last 1920 imgs) 1.0053 | run 1.0076 | 62 img/s | lr_head 1.00e-03 | gnorm max 13.38 (clip 1.0) | inf-steps 0 | amp scale 65536 | batch pos% [12.2 41.3 18.6] | mem 8.9GB | el 38s eta 252s
  ep1 60/237 (25%) loss(last 1856 imgs) 0.9778 | run 0.9932 | 61 img/s | lr_head 1.00e-03 | gnorm max 8.69 (clip 1.0) | inf-steps 0 | amp scale 65536 | batch pos% [13.5 43.9 19.4] | mem 8.9GB | el 69s eta 202s
  ep1 89/237 (38%) loss(last 1856 imgs) 0.9115 | run 0.

In [16]:

# 2) policy comparison on a subset
results = {p: run(p, EPOCHS=3, FRAC=0.4) for p in ["ignore"]}




===== policy=ignore | train 60691 imgs, 948 steps | val 39300 imgs =====
train masked entries/label : [2113 3066 9635]
train pos rate (unmasked)  : [0.128 0.423 0.185]
pos_weight                 : [6.84 1.36 4.4 ]
val certain pos / neg      : [ 4749 15920  6047] / [33128 21412 27286]  (tiny pos count = noisy metric)
  ep1 1/948 (0%) loss(last 64 imgs) 0.9540 | run 0.9540 | 0 img/s | lr_head 1.00e-03 | gnorm max 5.98 (clip 1.0) | inf-steps 0 | amp scale 65536 | batch pos% [ 4.8 37.3 18.5] | mem 8.9GB | el 1s eta 1296s
  ep1 30/948 (3%) loss(last 1856 imgs) 0.9874 | run 0.9863 | 61 img/s | lr_head 1.00e-03 | gnorm max 13.19 (clip 1.0) | inf-steps 0 | amp scale 65536 | batch pos% [10.6 41.2 18.1] | mem 8.9GB | el 32s eta 979s
  ep1 59/948 (6%) loss(last 1856 imgs) 0.9335 | run 0.9604 | 60 img/s | lr_head 1.00e-03 | gnorm max 10.09 (clip 1.0) | inf-steps 1 | amp scale 32768 | batch pos% [11.8 40.3 18.1] | mem 8.9GB | el 63s eta 947s
  ep1 88/948 (9%) loss(last 1856 imgs) 0.9262 | run 0.94